In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Machine Learning Baseline

The objective is to establish a Machine Learning baseline and demonstrate why classical algorithms (which require flattening spatial data into 1D vectors) are insufficient for classifying complex 2D SETI radio spectrograms.

Choosing the correct baseline model is a fair comparison. It is selected **Random Forest** for the following reasons, supported by existing literature on the SETI dataset:

- The dataset has 7 distinct classes. Random Forests can handle multi-class problems;
- SETI spectrograms contain heavy background radio noise. The ensemble nature of Random Forests (bagging and feature randomness) makes them resistant to overfitting on specific noise artifacts.

Before training, we must configure both our data pipeline and our model parameters to give ML the best possible chance of success:

- **PCA (`n_components=60`):** In our `ml_data_prep.py` pipeline, we reduced the 21,760 flattened pixels down to just 60 Principal Components. Because 90% of a SETI image is pure radio static, keeping high variance means keeping pure noise. Limiting to 60 components forces the algorithm to focus only on the strongest structures.
- **`n_estimators=150`:** We utilize 150 decision trees in our forest. This provides a large ensemble to stabilize predictions and capture complex decision boundaries without causing unnecessary computational overhead.
- **`class_weight='balanced'`:** To prevent the model from ignoring harder-to-classify signals, this parameter increases the penalty for misclassifying any minority or historically difficult classes during training.

In [ ]:
# Define classes in alphabetical order
CLASSES = ['brightpixel', 'narrowband', 'narrowbanddrd', 'noise', 
           'squarepulsednarrowband', 'squiggle', 'squigglesquarepulsednarrowband']

PROCESSED_ML_DIR = "../data/processed/ml"

X_train = np.load(os.path.join(PROCESSED_ML_DIR, 'X_train_pca.npy'))
X_valid = np.load(os.path.join(PROCESSED_ML_DIR, 'X_valid_pca.npy'))
y_train = np.load(os.path.join(PROCESSED_ML_DIR, 'y_train.npy'))
y_valid = np.load(os.path.join(PROCESSED_ML_DIR, 'y_valid.npy'))

print(f"Training Features Shape: {X_train.shape} (Images, PCA Components)")
print(f"Validation Features Shape:  {X_test.shape}")

The output above confirms that our custom DVC data pipeline successfully loaded, scaled, and transformed the raw image dataset into a format suitable for ML. 

- **Class Balance (`3500` Train / `700` Validation):** We loaded an evenly distributed dataset (500 training samples and 100 validation samples per class). Classical algorithms, particularly decision trees, are highly sensitive to majority-class bias. This ensures our baseline metrics reflect learning rather than class imbalances.

In [ ]:
rf_model = RandomForestClassifier(n_estimators = 150, random_state = 42, n_jobs = -1, class_weight = 'balanced')
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_valid)

accuracy = accuracy_score(y_valid, y_pred)
print(f"Baseline ML Accuracy: {accuracy * 100:.2f}%")

At first look, an accuracy of **28.86%** might appear low, but evaluating it requires context:

- This is the limitation of classical ML on image data. By flattening a 2D spectrogram into a 1D vector, we destroy the spatial and temporal topology of the signal. A decision tree evaluates features (pixels) independently and cannot easily understand the slope of a Doppler drifted signal (`narrowbanddrd`) or the wave-like curve of a `squiggle`. 
- This baseline proves that we cannot solve this problem with independent feature splits.

In [ ]:
print(classification_report(y_valid, y_pred, target_names = CLASSES))

While the global accuracy gives us a high-level overview, the **Classification Report** reveals exactly where and why the ML model fails. By analyzing the **Recall** and **F1-scores** across different classes, a clear pattern emerges:

1. The model is good at finding pure radio noise, correctly identifying 88% of all noise samples. This makes sense: pure noise has high entropy and lacks spatial structure. A Random Forest can easily identify this lack of structure even when the image is flattened into a 1D array.
2. `narrowband` (4% recall), `narrowbanddrd` (3% recall), and `squiggle` (12% recall) are defined by their continuous 2D shapes (straight vertical lines, diagonal drifted lines, and sine waves). Because the ML pipeline flattened images into 1D vectors, the algorithm lost the 2D spatial context.
3. The macro-average F1-score is only **0.24**. This is the harmonic mean of precision and recall, meaning a low score indicates the model is suffering from both high False Positives and high False Negatives across the board for structured signals.

In [ ]:
# Generate Confusion Matrix
cm = confusion_matrix(y_valid, y_pred)

plt.figure(figsize = (10, 7))
sns.heatmap(cm, annot = True, fmt = 'd', cmap = 'Blues', xticklabels = CLASSES, yticklabels = CLASSES, cbar = False)
plt.title(f"ML Baseline Confusion Matrix (Accuracy: {accuracy * 100:.1f}%)", fontsize = 14, pad = 15)
plt.ylabel('True Class', fontsize = 12, fontweight = 'bold')
plt.xlabel('Predicted Class', fontsize = 12, fontweight = 'bold')
plt.xticks(rotation = 45, ha = 'right')
plt.tight_layout()
plt.show()

The confusion matrix provides a look at exactly how this model is misclassifying the signals.

In [ ]:
# Extract metrics dynamically for visual analysis
report_dict = classification_report(y_valid, y_pred, target_names = CLASSES, output_dict = True)
recalls = [report_dict[cls]['recall'] * 100 for cls in CLASSES]

# Plotting Recall (Sensitivity) per class to show exactly where the model fails
plt.figure(figsize = (10, 5))
bars = plt.bar(CLASSES, recalls, color = sns.color_palette("rocket", len(CLASSES)))
plt.axhline(y = 14.28, color = 'red', linestyle = '--', label = "Random Chance (14.28%)")

plt.title("Model Recall (Sensitivity) per Class", fontsize = 14)
plt.ylabel("Recall (%)", fontsize = 12)
plt.xticks(rotation = 45, ha = 'right')
plt.ylim(0, 100)
plt.legend()

# Add percentage labels on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f'{yval:.1f}%', ha = 'center', va = 'bottom', fontsize = 10)

plt.tight_layout()
plt.show()

The bar chart above maps the sensitivity of our model across all classes. The red line represents the threshold of random guessing (14.28%). This visualization exposes a severe underlying flaw in how the model processes image data.